In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.coordinates import ICRS, Galactic, FK4, FK5
import numpy as np
import matplotlib.pyplot as plt
from astropy.coordinates import angular_separation
from astropy.wcs import WCS
import gc
from matplotlib.gridspec import GridSpec
from astropy import units as u

In [ ]:
hdu = fits.open('/srv/data/gmims/gmims-hbn/GMIMS-HBN_v1_gal_car_freq_IQU.fits')

data = hdu[0].data
hdr  = hdu[0].header

I  = data[0]
Q  = data[1]
U  = data[2]
PI = np.sqrt(Q**2 + U**2)

print(data.shape)
#print(repr(hdr))

wcs = WCS(hdr).dropaxis(3).dropaxis(2)
print(wcs)

In [ ]:
hdu

In [ ]:
freqs = WCS(hdr).dropaxis(3).all_pix2world(0,0,range(WCS(hdr).dropaxis(3).array_shape[0]),0)[2]/1e6
b_arr = WCS(hdr).dropaxis(3).all_pix2world(0,range(WCS(hdr).dropaxis(3).array_shape[1]),0,0)[1]
l_arr = WCS(hdr).dropaxis(3).all_pix2world(range(WCS(hdr).dropaxis(3).array_shape[2]),0,0,0)[0]


In [ ]:
def leakage_plot(l0, b0, lb_wid, fidx, src, vmax=50):

    fig = plt.figure(figsize=(12,7))
    gs = GridSpec(3, 4, figure=fig, height_ratios=[1,1,1])

    cx = SkyCoord([l0+lb_wid/2, l0-lb_wid/2], [b0, b0], frame=Galactic, unit="deg")
    cy = SkyCoord([l0, l0], [b0-lb_wid/2, b0+lb_wid/2], frame=Galactic, unit="deg")

    lidx = abs(l0 - l_arr).argmin()
    bidx = abs(b0 - b_arr).argmin()

    fs = 9

    ax1 = fig.add_subplot(gs[0,0], projection=wcs.celestial)
    ax2 = fig.add_subplot(gs[0,1], projection=wcs.celestial)
    ax3 = fig.add_subplot(gs[0,2], projection=wcs.celestial)
    ax4 = fig.add_subplot(gs[0,3], projection=wcs.celestial)
    ax6 = fig.add_subplot(gs[1,1], projection=wcs.celestial)
    ax7 = fig.add_subplot(gs[1,2], projection=wcs.celestial)
    ax8 = fig.add_subplot(gs[1,3], projection=wcs.celestial)
    ax9 = fig.add_subplot(gs[2,0])
    ax10= fig.add_subplot(gs[2,1])
    ax11= fig.add_subplot(gs[2,2])
    ax12= fig.add_subplot(gs[2,3])

    plt.subplots_adjust(wspace=0.3,hspace=0.3,bottom=0.05,left=0.05,top=0.95,right=0.95) 
    ims = []
    ims.append(ax1.imshow(I[fidx], origin='lower',vmin=0, vmax=vmax,cmap='viridis'))
    ims.append(ax2.imshow(PI[fidx],origin='lower',vmin=0, vmax=vmax/20,cmap='viridis'))
    ims.append(ax3.imshow(Q[fidx], origin='lower',vmin=-vmax/20, vmax=vmax/20,cmap='rainbow'))
    ims.append(ax4.imshow(U[fidx], origin='lower',vmin=-vmax/20, vmax=vmax/20,cmap='rainbow'))

    ims.append(ax6.imshow(PI[fidx]/I[fidx],origin='lower',vmin=0, vmax=0.5,cmap='viridis'))
    ims.append(ax7.imshow(Q[fidx]/I[fidx], origin='lower',vmin=-0.5, vmax=0.5,cmap='rainbow'))
    ims.append(ax8.imshow(U[fidx]/I[fidx], origin='lower',vmin=-0.5, vmax=0.5,cmap='rainbow'))

    ax9.plot(l_arr,  I[fidx][bidx])
    ax10.plot(l_arr, PI[fidx][bidx])
    ax11.plot(l_arr, Q[fidx][bidx])
    ax12.plot(l_arr, U[fidx][bidx])

    ax10_frac = ax10.twinx()
    ax11_frac = ax11.twinx()
    ax12_frac = ax12.twinx()

    ax10_frac.plot(l_arr, PI[fidx][bidx]/I[fidx][bidx],color='k')
    ax10_frac.axhline(y=0.03,color='blue',linestyle='dashed')
    ax11_frac.plot(l_arr, Q[fidx][bidx]/I[fidx][bidx],color='k')
    ax12_frac.plot(l_arr, U[fidx][bidx]/I[fidx][bidx],color='k')

    ax9.set_ylabel(r'Stokes $I$ (K)',fontsize=fs)
    ax10.set_ylabel(r'PI (K)',fontsize=fs)
    ax11.set_ylabel(r'Stokes $Q$ (K)',fontsize=fs)
    ax12.set_ylabel(r'Stokes $U$ (K)',fontsize=fs)

    ax10_frac.set_ylabel(r'PI/$I$ (K)',fontsize=fs)
    ax11_frac.set_ylabel(r'$Q$/$I$',fontsize=fs)
    ax12_frac.set_ylabel(r'$U$/$I$',fontsize=fs)
    
    for ax in [ax10_frac, ax11_frac, ax12_frac]:
        ax.tick_params(labelsize=fs)

    for ax in [ax1, ax2, ax3, ax4, ax6, ax7, ax8]:
        ax.set_xlim(wcs.world_to_pixel(cx)[0])
        ax.set_ylim(wcs.world_to_pixel(cy)[1])
        ax.set_xlabel(' ')
        ax.set_ylabel(' ')
        ax.tick_params(labelsize=fs)

    for ax in [ax9, ax10, ax11, ax12]:
        ax.set_xlim(l0+lb_wid/2, l0-lb_wid/2)
        ax.tick_params(labelsize=fs)
    ax9.set_ylim(0,vmax)
    ax10.set_ylim(0,vmax/20)
    for ax in [ax11, ax12]:
        ax.set_ylim(-vmax/20,vmax/20)
    ax10_frac.set_ylim(0,0.5)
    for ax in [ax11_frac, ax12_frac]:
        ax.set_ylim(-0.5,0.5)

    def add_colorbar(im, ax):
        cbar = plt.colorbar(im, ax=ax, fraction=0.048, pad=0.01)
        cbar.ax.tick_params(labelsize=fs)
        return cbar

    axs = [ax1, ax2, ax3, ax4, ax6, ax7, ax8]
    for i in range(0,7):
        add_colorbar(ims[i], axs[i])

    ax1.set_title(src+r' Stokes $I$',fontsize=fs)
    ax2.set_title(src+r' PI',fontsize=fs)
    ax3.set_title(src+r' Stokes $Q$',fontsize=fs)
    ax4.set_title(src+r' Stokes $U$',fontsize=fs)
    ax6.set_title(src+r' PI/$I$',fontsize=fs)
    ax7.set_title(src+r' $Q$/$I$',fontsize=fs)
    ax8.set_title(src+r' $U$/$I$',fontsize=fs)

    #def adjust_bottom_panels(axs, shrink_factor=0.8):
    for ax in [ax9, ax10, ax11, ax12]:
        left, bottom, width, height = ax.get_position().bounds
        new_width = width * 0.8
        x_offset = (width - new_width) / 2
        ax.set_position([left + x_offset+0.009, bottom, new_width, height])

    #adjust_bottom_panels([ax9, ax10, ax11, ax12], shrink_factor=0.)

        
    return

In [ ]:
l_arr[l_arr>359] = np.nan

# Cyg A
#src = 'Cyg A'
#ra  = 19+59/60+28.3/3600
#dec = 40+44/60+2/3600

# Tau A
#src = 'Tau A'
#ra  = 5+34/60+31.97/3600
#dec = 22+0/60+52.1/3600

# Cas A
#src = 'Cas A'
#ra  = 23+23/60+28/3600
#dec = 58+49/60+0/3600

# IC443
#src = 'IC 443'
#ra  = 6+17/60+13/3600
#dec = 22+31/60+5/3600

# Sh 131
src = 'Sh 131'
cl = 99.3
cb = 3.7

c = SkyCoord(ra=ra*u.hour, dec=dec*u.degree, frame='icrs')
print(c.galactic.l.deg)
print(c.galactic.b.deg)

In [ ]:
fpick = 1420.
fidx  = abs(fpick-freqs).argmin()
print(freqs[fidx])
print(fidx)

leakage_plot(c.galactic.l.deg, c.galactic.b.deg+0.2, 8, fidx, src, vmax=10)
#plt.savefig('../plots/leakage_CygA.png')
#leakage_plot(c.galactic.l.deg, c.galactic.b.deg, 8, fidx, vmax=7)

In [ ]:
fpick = 1420.
fidx  = abs(fpick-freqs).argmin()
print(freqs[fidx])
print(fidx)

leakage_plot(cl, cb, 8, fidx, src, vmax=4)
#plt.savefig('../plots/leakage_CygA.png')


In [ ]:
0.45/8.5

In [ ]:
2.5/50